In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/TFM/'
MODELOS = BASE + 'modelos/'

import joblib
import numpy as np

# Modelo CO2
model_co2 = joblib.load(MODELOS + 'model_co2.pkl')
print("Features modelo CO2:")
print(f"Número: {model_co2.n_features_in_}")
print(f"Nombres: {list(model_co2.feature_names_in_)}")

# Scaler
scaler = joblib.load(MODELOS + 'scaler.pkl')
print(f"\nFeatures scaler: {scaler.n_features_in_}")

# KMeans
model_clusters = joblib.load(MODELOS + 'model_clusters.pkl')
print(f"\nFeatures KMeans: {model_clusters.n_features_in_}")

# Series temporales
model_series = joblib.load(MODELOS + 'model_series.pkl')
print(f"\nTipo modelo series: {type(model_series)}")
print(f"Contenido: {model_series}")

Mounted at /content/drive
Features modelo CO2:
Número: 43
Nombres: [np.str_('reportingYear'), np.str_('Latitude'), np.str_('Longitude'), np.str_('WASTE_Recovery_HW'), np.str_('AIR_Nitrogen oxides (NOX)'), np.str_('WASTE_Disposal_HW'), np.str_('WASTE_Recovery_NONHW'), np.str_('WASTE_Disposal_NONHW'), np.str_('EPRTR_SectorName_Chemical industry'), np.str_('EPRTR_SectorName_Energy sector'), np.str_('EPRTR_SectorName_Mineral industry'), np.str_('EPRTR_SectorName_Other activities'), np.str_('EPRTR_SectorName_Paper and wood production and processing'), np.str_('EPRTR_SectorName_Production and processing of metals'), np.str_('EPRTR_SectorName_Waste and wastewater management'), np.str_('countryName_Belgium'), np.str_('countryName_Bulgaria'), np.str_('countryName_Croatia'), np.str_('countryName_Cyprus'), np.str_('countryName_Czechia'), np.str_('countryName_Denmark'), np.str_('countryName_Estonia'), np.str_('countryName_Finland'), np.str_('countryName_France'), np.str_('countryName_Germany'), np

In [ ]:
scaler = joblib.load(MODELOS + 'scaler.pkl')
print("Orden de features del scaler:")
for i, name in enumerate(scaler.feature_names_in_):
    print(f"{i}: {name}")

Orden de features del scaler:
0: AIR_Ammonia (NH3)
1: AIR_Carbon dioxide (CO2)
2: AIR_Carbon dioxide (CO2) excluding biomass
3: AIR_Carbon monoxide (CO)
4: AIR_Chlorine and inorganic compounds (as HCl)
5: AIR_Hydrochlorofluorocarbons (HCFCs)
6: AIR_Mercury and compounds (as Hg)
7: AIR_Methane (CH4)
8: AIR_Nickel and compounds (as Ni)
9: AIR_Nitrogen oxides (NOX)
10: AIR_Nitrous oxide (N2O)
11: AIR_Non-methane volatile organic compounds (NMVOC)
12: AIR_Particulate matter (PM10)
13: AIR_Sulphur oxides (SOX)
14: AIR_Zinc and compounds (as Zn)
15: WASTE_Disposal_HW
16: WASTE_Disposal_NONHW
17: WASTE_Recovery_HW
18: WASTE_Recovery_NONHW
19: WATER_Arsenic and compounds (as As)
20: WATER_Chlorides (as total Cl)
21: WATER_Copper and compounds (as Cu)
22: WATER_Fluorides (as total F)
23: WATER_Lead and compounds (as Pb)
24: WATER_Nickel and compounds (as Ni)
25: WATER_Total nitrogen
26: WATER_Total organic carbon(as total C or COD/3) (TOC)
27: WATER_Total phosphorus
28: WATER_Zinc and compounds

In [ ]:
# Cargar las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import os

# Cargar el dataset
DATA = BASE + 'datasets/'
df_merged = pd.read_csv(DATA + 'dataset_merged_limpio.csv', low_memory=False)

# Las mismas columnas de contaminantes que usó el clustering
POLLUTANT_COLS = [c for c in df_merged.columns
                  if c.startswith(('AIR_', 'WATER_', 'WASTE_', 'TRANSFER_'))]

print(f"Columnas de contaminantes: {len(POLLUTANT_COLS)}")

Columnas de contaminantes: 35


In [ ]:
# Recrear el panel país-año igual que en el notebook de clustering
panel = df_merged.groupby(['countryName', 'reportingYear'])[POLLUTANT_COLS].sum(min_count=1)
panel_nfac = df_merged.groupby(['countryName', 'reportingYear'])['FacilityInspireId'].nunique().rename('n_facilities')
panel_norm = panel.join(panel_nfac)
panel_norm[POLLUTANT_COLS] = panel_norm[POLLUTANT_COLS].fillna(0).div(panel_norm['n_facilities'], axis=0)

# Agregar a nivel país
country_df = panel_norm.groupby('countryName')[POLLUTANT_COLS].mean()

# Crear y guardar el scaler
X_log = np.log1p(country_df)
scaler_clusters = StandardScaler()
scaler_clusters.fit(X_log)
joblib.dump(scaler_clusters, MODELOS + 'scaler_clusters.pkl')

size = os.path.getsize(MODELOS + 'scaler_clusters.pkl') / 1024
print(f"✓ scaler_clusters.pkl guardado — {size:.1f} KB")

✓ scaler_clusters.pkl guardado — 2.8 KB
